In [8]:
import pandas as pd
import numpy as np
import psycopg

In [9]:
conn = psycopg.connect("dbname=dailyedge_development")

print("Connected to dailyedge_development")

Connected to dailyedge_development


In [10]:
query = """
SELECT
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    COUNT(*) AS total_rows
FROM CANDLES;
"""

db_info = pd.read_sql(query, conn)
db_info

/tmp/ipykernel_3468/1321250083.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_info = pd.read_sql(query, conn)


,first_timestamp,last_timestamp,total_rows
0,2008-12-11 01:38:00,2026-08-18 02:36:00,5898467


In [11]:
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM CANDLES
WHERE timestamp >= '2025-09-01 08:30:00'
  AND timestamp <= '2026-07-07 15:15:00'
  AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
ORDER BY timestamp;
"""

candles = pd.read_sql(query, conn)

print("Rows:", len(candles))
print("First timestamp:", candles["timestamp"].min())
print("Last timestamp:", candles["timestamp"].max())

/tmp/ipykernel_3468/464963839.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql(query, conn)


Rows: 87179
First timestamp: 2025-09-01 08:30:00
Last timestamp: 2026-07-07 15:15:00


In [12]:
candles["Date"] = candles["timestamp"].dt.date
candles["Day"] = candles["timestamp"].dt.day_name()

session_counts = (
    candles.groupby(["Date", "Day"])
    .size()
    .reset_index(name="Candles")
)

print("Sessions:", len(session_counts))
print()
print(session_counts["Day"].value_counts().sort_index())

Sessions: 219

Day
Friday       43
Monday       45
Thursday     42
Tuesday      45
Wednesday    44
Name: count, dtype: int64


In [13]:
def evaluate_raw_cleanliness(session, stop=35, target=70):
    session = session.sort_values("timestamp").reset_index(drop=True)

    opening_price = session.iloc[0]["open"]

    upper_target = opening_price + target
    lower_target = opening_price - target

    upper_hits = session.index[session["high"] >= upper_target]
    lower_hits = session.index[session["low"] <= lower_target]

    if len(upper_hits) == 0 and len(lower_hits) == 0:
        return "Neither", None

    if len(upper_hits) == 0:
        direction = "Short"
        target_idx = lower_hits[0]
    elif len(lower_hits) == 0:
        direction = "Long"
        target_idx = upper_hits[0]
    else:
        first_upper = upper_hits[0]
        first_lower = lower_hits[0]

        if first_upper == first_lower:
            return "Unknown", "Unknown"

        if first_upper < first_lower:
            direction = "Long"
            target_idx = first_upper
        else:
            direction = "Short"
            target_idx = first_lower

    before_target = session.loc[:target_idx]

    if direction == "Long":
        adverse_level = opening_price - stop

        adverse_hits = before_target.index[
            before_target["low"] <= adverse_level
        ]

    else:
        adverse_level = opening_price + stop

        adverse_hits = before_target.index[
            before_target["high"] >= adverse_level
        ]

    if len(adverse_hits) == 0:
        cleanliness = "Clean"
    else:
        first_adverse = adverse_hits[0]

        if first_adverse == target_idx:
            cleanliness = "Unknown"
        else:
            cleanliness = "Not Clean"

    return direction, cleanliness

In [14]:
stop_target_pairs = [
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 80),
    (50, 100),
    (75, 100),
    (75, 150),
    (100, 150)
]

all_raw_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        direction, cleanliness = evaluate_raw_cleanliness(
            session,
            stop=stop,
            target=target
        )

        all_raw_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Direction": direction,
            "Cleanliness": cleanliness
        })

all_raw_results = pd.DataFrame(all_raw_results)

print("Rows:", len(all_raw_results))
print()
print(
    all_raw_results
    .groupby(["Stop", "Target"])["Cleanliness"]
    .value_counts(dropna=False)
)

Rows: 1971

Stop  Target  Cleanliness
15    25      Clean          139
              Unknown         53
              Not Clean       26
              NaN              1
25    50      Clean          149
              Not Clean       60
              Unknown          7
              NaN              3
35    70      Clean          144
              Not Clean       69
              NaN              6
50    70      Clean          174
              Not Clean       38
              NaN              6
              Unknown          1
      80      Clean          159
              Not Clean       53
              NaN              7
      100     Clean          140
              Not Clean       68
              NaN             11
75    100     Clean          181
              Not Clean       27
              NaN             11
      150     Clean          129
              Not Clean       50
              NaN             40
100   150     Clean          151
              NaN             40
     

In [15]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

resolved_raw = all_raw_results[
    all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
].copy()

resolved_raw["Clean"] = resolved_raw["Cleanliness"] == "Clean"

raw_weekday_summary = (
    resolved_raw
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Clean", "size"),
        Clean=("Clean", "sum")
    )
    .reset_index()
)

raw_weekday_summary["Clean Rate"] = (
    raw_weekday_summary["Clean"]
    / raw_weekday_summary["Resolved"]
    * 100
)

raw_cleanliness_table = (
    raw_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Clean Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved = (
    all_raw_results[
        ~all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

raw_cleanliness_table["Unresolved"] = unresolved

raw_cleanliness_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       90.62    83.33      83.78     79.31   83.87          54
25   50       69.05    77.27      77.27     74.36   57.50          10
35   70       72.09    77.78      68.18     65.85   52.50           6
50   70       86.05    84.44      79.55     80.49   79.49           7
     80       83.72    77.78      68.18     77.50   67.50           7
     100      73.81    63.64      60.47     72.50   66.67          11
75   100      92.86    77.27      90.70     90.00   84.62          11
     150      81.82    55.88      80.49     67.57   73.53          40
100  150      90.91    79.41      87.80     75.68   88.24          40

In [16]:
def evaluate_continuous_trail(
    rth,
    opening_price,
    target_distance,
    trail_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)
            new_stop = new_highest - trail_distance

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            # Low first
            if old_stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif new_stop_hit:
                high_first = "Failure"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if {low_first, high_first} == {"Continue", "Failure"}:
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if low_first == "Continue" and high_first == "Continue":
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + trail_distance

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            # High first
            if old_stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif new_stop_hit:
                low_first = "Failure"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if {high_first, low_first} == {"Continue", "Failure"}:
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if high_first == "Continue" and low_first == "Continue":
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [17]:
def get_first_target(rth, opening_price, target_distance):
    upper_target = opening_price + target_distance
    lower_target = opening_price - target_distance

    target_hits = rth[
        (rth["high"] >= upper_target) |
        (rth["low"] <= lower_target)
    ]

    if target_hits.empty:
        return "Neither", None

    first_hit = target_hits.iloc[0]

    hit_upper = first_hit["high"] >= upper_target
    hit_lower = first_hit["low"] <= lower_target

    if hit_upper and hit_lower:
        return "Ambiguous", first_hit["timestamp"]

    result = "Long" if hit_upper else "Short"

    return result, first_hit["timestamp"]

In [18]:
all_trail_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_continuous_trail(
            session,
            opening_price,
            target_distance=target,
            trail_distance=stop
        )

        all_trail_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_trail_results = pd.DataFrame(all_trail_results)

print("Rows:", len(all_trail_results))
print()
print(
    all_trail_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1971

Stop  Target  Outcome  
15    25      Success      127
              Unknown       39
              Failure       36
              Ambiguous     16
              Neither        1
25    50      Failure      108
              Success       97
              Unknown        9
              Neither        3
              Ambiguous      2
35    70      Failure      131
              Success       80
              Neither        6
              Unknown        2
50    70      Success      120
              Failure       92
              Neither        6
              Unknown        1
      80      Failure      111
              Success       99
              Neither        7
              Unknown        2
      100     Failure      145
              Success       61
              Neither       11
              Unknown        2
75    100     Success      113
              Failure       93
              Neither       11
              Unknown        2
      150     Failure      118
   

In [19]:
resolved_trail = all_trail_results[
    all_trail_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_trail["Success"] = resolved_trail["Outcome"] == "Success"

trail_weekday_summary = (
    resolved_trail
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

trail_weekday_summary["Survival Rate"] = (
    trail_weekday_summary["Success"]
    / trail_weekday_summary["Resolved"]
    * 100
)

trail_survival_table = (
    trail_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_trail = (
    all_trail_results[
        ~all_trail_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

trail_survival_table["Unresolved"] = unresolved_trail

trail_survival_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       83.87    75.00      77.78     75.86   77.42          56
25   50       38.10    51.16      53.49     51.35   42.50          14
35   70       39.53    40.91      37.21     36.59   35.00           8
50   70       62.79    60.00      54.55     58.54   46.15           7
     80       55.81    51.16      47.73     50.00   30.00           9
     100      41.46    25.58      23.26     30.00   28.21          13
75   100      76.19    51.16      48.84     53.85   43.59          13
     150      42.42    32.35      26.83     43.24   26.47          40
100  150      57.58    47.06      56.10     56.76   52.94          40

In [20]:
def evaluate_one_move_breakeven(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        initial_stop = opening_price - stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = high >= threshold

            # Low first
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                if low <= opening_price:
                    high_first = "Failure"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        initial_stop = opening_price + stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                if high >= opening_price:
                    low_first = "Failure"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [21]:
all_be_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_one_move_breakeven(
            session,
            opening_price,
            target_distance=target,
            stop_distance=stop
        )

        all_be_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_be_results = pd.DataFrame(all_be_results)

print("Rows:", len(all_be_results))
print()
print(
    all_be_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1971

Stop  Target  Outcome  
15    25      Success      139
              Unknown       38
              Failure       25
              Ambiguous     16
              Neither        1
25    50      Success      141
              Failure       68
              Unknown        5
              Neither        3
              Ambiguous      2
35    70      Success      145
              Failure       67
              Neither        6
              Unknown        1
50    70      Success      182
              Failure       30
              Neither        6
              Unknown        1
      80      Success      160
              Failure       51
              Neither        7
              Unknown        1
      100     Success      128
              Failure       79
              Neither       11
              Unknown        1
75    100     Success      172
              Failure       35
              Neither       11
              Unknown        1
      150     Success      112
   

In [22]:
resolved_be = all_be_results[
    all_be_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_be["Success"] = resolved_be["Outcome"] == "Success"

be_weekday_summary = (
    resolved_be
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

be_weekday_summary["Survival Rate"] = (
    be_weekday_summary["Success"]
    / be_weekday_summary["Resolved"]
    * 100
)

be_survival_table = (
    be_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_be = (
    all_be_results[
        ~all_be_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

be_survival_table["Unresolved"] = unresolved_be

be_survival_table.round(2)


Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       87.50    82.86      81.08     82.76   90.32          55
25   50       66.67    68.18      70.45     74.36   57.50          10
35   70       72.09    68.89      68.18     73.17   58.97           7
50   70       90.70    88.89      86.36     90.24   71.79           7
     80       86.05    77.27      77.27     82.50   55.00           8
     100      78.57    60.47      58.14     65.00   46.15          12
75   100      88.10    86.05      79.07     80.00   82.05          12
     150      60.61    61.76      58.54     64.86   67.65          40
100  150      69.70    76.47      68.29     78.38   88.24          40

In [23]:
# Load evaluated First-to-100 predictions

predictions = pd.read_csv("../data/nq_first_to_100_predictions_evaluated.csv")

predictions["Date"] = pd.to_datetime(predictions["Date"])

print("Prediction rows:", len(predictions))
print("Date range:", predictions["Date"].min(), "to", predictions["Date"].max())

predictions[["Date", "Day", "Bias", "Result", "Correct"]].head()

Prediction rows: 219
Date range: 2025-09-01 00:00:00 to 2026-07-07 00:00:00


,Date,Day,Bias,Result,Correct
0,2025-09-01,Monday,Long,Invalid,NaN
1,2025-09-02,Tuesday,Short,Long,False
2,2025-09-03,Wednesday,Long,Long,True
3,2025-09-04,Thursday,Short,Long,False
4,2025-09-05,Friday,Long,Short,False


In [24]:
# Check date overlap between predictions and cleanliness results

prediction_dates = set(
    predictions["Date"].dropna().dt.date
)

cleanliness_dates = set(
    pd.to_datetime(all_be_results["Date"]).dropna().dt.date
)

matched_dates = prediction_dates & cleanliness_dates
prediction_only = prediction_dates - cleanliness_dates
cleanliness_only = cleanliness_dates - prediction_dates

print("Prediction dates:", len(prediction_dates))
print("Cleanliness dates:", len(cleanliness_dates))
print("Matched dates:", len(matched_dates))

print("\nPrediction-only dates:", sorted(prediction_only))
print("\nCleanliness-only dates:", sorted(cleanliness_only))

Prediction dates: 219
Cleanliness dates: 219
Matched dates: 218

Prediction-only dates: [datetime.date(2026, 4, 3)]

Cleanliness-only dates: [datetime.date(2026, 4, 24)]


In [25]:
# Prediction accuracy population for matched dates

matched_predictions = predictions[
    predictions["Date"].dt.date.isin(matched_dates)
].copy()

print("Matched predictions:", len(matched_predictions))

print("\nCorrect distribution:")
print(matched_predictions["Correct"].value_counts(dropna=False))

print("\nResult distribution:")
print(matched_predictions["Result"].value_counts(dropna=False))

Matched predictions: 218

Correct distribution:
Correct
True     118
False     87
NaN       13
Name: count, dtype: int64

Result distribution:
Result
Short      106
Long        99
Invalid      9
Neither      4
Name: count, dtype: int64


In [26]:
# Merge resolved prediction accuracy with BE + Add cleanliness results

resolved_predictions = matched_predictions[
    matched_predictions["Correct"].notna()
].copy()

# Standardize Date dtype before merging
all_be_results["Date"] = pd.to_datetime(all_be_results["Date"])

be_accuracy = all_be_results.merge(
    resolved_predictions[["Date", "Correct"]],
    on="Date",
    how="inner"
)

print("Resolved prediction dates:", len(resolved_predictions))
print("Merged rows:", len(be_accuracy))
print("Unique merged dates:", be_accuracy["Date"].nunique())

be_accuracy.head()

Resolved prediction dates: 205
Merged rows: 1845
Unique merged dates: 205


,Date,Day,Stop,Target,Outcome,Correct
0,2025-09-02,Tuesday,15,25,Success,False
1,2025-09-03,Wednesday,15,25,Success,True
2,2025-09-04,Thursday,15,25,Success,False
3,2025-09-05,Friday,15,25,Unknown,False
4,2025-09-08,Monday,15,25,Success,True


In [27]:
# BE + Add success rate by weekday — correct predictions only

correct_be = be_accuracy[
    (be_accuracy["Correct"] == True) &
    (be_accuracy["Outcome"].isin(["Success", "Failure"]))
].copy()

correct_success_table = (
    correct_be
    .assign(Success=correct_be["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Success"]
    .mean()
    .mul(100)
    .unstack("Day")
)

correct_success_table = correct_success_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

correct_success_table.index = [
    f"{stop}/{target}"
    for stop, target in correct_success_table.index
]

correct_success_table.index.name = "Pair"

correct_success_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,90.91,85.00,77.27,91.67,89.47
25/50,77.78,62.96,68.00,76.47,64.00
35/70,78.95,64.29,64.00,63.16,69.23
50/70,89.47,85.71,76.00,89.47,76.92
50/80,89.47,74.07,68.00,84.21,55.56
50/100,84.21,59.26,52.00,63.16,44.44
75/100,89.47,85.19,76.00,68.42,85.19
75/150,60.00,66.67,62.50,55.56,66.67
100/150,66.67,85.71,70.83,83.33,87.50


In [28]:
# Actual hit rate by weekday:
# prediction must be correct AND BE + Add trade must succeed

weekday_accuracy = (
    resolved_predictions
    .groupby("Day")["Correct"]
    .mean()
)

actual_hit_rate = correct_success_table.copy()

for day in actual_hit_rate.columns:
    actual_hit_rate[day] = (
        actual_hit_rate[day] * weekday_accuracy[day]
    )

actual_hit_rate.index.name = "Pair"

print("Prediction accuracy used:")
display((weekday_accuracy * 100).round(2))

print("\nActual hit rate (%):")
display(actual_hit_rate.round(2))

Prediction accuracy used:


Day
Friday       72.97
Monday       46.34
Thursday      47.5
Tuesday      63.64
Wednesday    58.14
Name: Correct, dtype: object


Actual hit rate (%):


Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,42.13,54.09,44.93,43.54,65.29
25/50,36.04,40.07,39.53,36.32,46.70
35/70,36.59,40.91,37.21,30.00,50.52
50/70,41.46,54.55,44.19,42.50,56.13
50/80,41.46,47.14,39.53,40.00,40.54
50/100,39.02,37.71,30.23,30.00,32.43
75/100,41.46,54.21,44.19,32.50,62.16
75/150,27.80,42.42,36.34,26.39,48.65
100/150,30.89,54.55,41.18,39.58,63.85


In [29]:
# Continuous trail success rate by weekday — correct predictions only

all_trail_results["Date"] = pd.to_datetime(all_trail_results["Date"])

trail_accuracy = all_trail_results.merge(
    resolved_predictions[["Date", "Correct"]],
    on="Date",
    how="inner"
)

correct_trail = trail_accuracy[
    (trail_accuracy["Correct"] == True) &
    (trail_accuracy["Outcome"].isin(["Success", "Failure"]))
].copy()

trail_success_table = (
    correct_trail
    .assign(Success=correct_trail["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Success"]
    .mean()
    .mul(100)
    .unstack("Day")
)

trail_success_table = trail_success_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

trail_success_table.index = [
    f"{stop}/{target}"
    for stop, target in trail_success_table.index
]

trail_success_table.index.name = "Pair"

trail_success_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,81.82,76.19,71.43,91.67,68.42
25/50,50.00,42.31,50.00,50.00,44.00
35/70,52.63,35.71,37.50,31.58,44.44
50/70,68.42,53.57,48.00,52.63,57.69
50/80,63.16,48.15,40.00,52.63,33.33
50/100,44.44,22.22,24.00,26.32,29.63
75/100,78.95,48.15,40.00,44.44,44.44
75/150,46.67,38.10,29.17,38.89,16.67
100/150,60.00,52.38,58.33,50.00,50.00


In [30]:
all_be_results["Outcome"].value_counts(dropna=False)

Outcome
Success      1315
Failure       465
Neither       125
Unknown        48
Ambiguous      18
Name: count, dtype: int64

In [31]:
def evaluate_one_move_breakeven_no_add(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        stop = opening_price - stop_distance
        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = high >= threshold

            # Low first
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                if low <= opening_price:
                    high_first = "Breakeven"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first in ["Success", "Failure", "Breakeven"]:
                    return low_first

            if low_first != high_first and {
                low_first,
                high_first
            } <= {"Success", "Failure", "Breakeven"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        stop = opening_price + stop_distance
        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                if high >= opening_price:
                    low_first = "Breakeven"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first in ["Success", "Failure", "Breakeven"]:
                    return high_first

            if high_first != low_first and {
                high_first,
                low_first
            } <= {"Success", "Failure", "Breakeven"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [32]:
all_be_no_add_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_one_move_breakeven_no_add(
            session,
            opening_price,
            target_distance=target,
            stop_distance=stop
        )

        all_be_no_add_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_be_no_add_results = pd.DataFrame(all_be_no_add_results)

print("Rows:", len(all_be_no_add_results))
print()
print(
    all_be_no_add_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1971

Stop  Target  Outcome  
15    25      Success      139
              Unknown       49
              Ambiguous     16
              Breakeven     11
              Failure        3
              Neither        1
25    50      Success      141
              Breakeven     42
              Unknown       16
              Failure       15
              Neither        3
              Ambiguous      2
35    70      Success      145
              Breakeven     36
              Failure       26
              Neither        6
              Unknown        6
50    70      Success      182
              Breakeven     18
              Failure       11
              Neither        6
              Unknown        2
      80      Success      160
              Breakeven     26
              Failure       23
              Neither        7
              Unknown        3
      100     Success      128
              Breakeven     49
              Failure       28
              Neither       11
   

In [33]:
def evaluate_predicted_one_move_breakeven(
    rth,
    opening_price,
    direction,
    target_distance,
    stop_distance
):
    # -------------------------
    # LONG
    # -------------------------

    if direction == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance
        stop = opening_price - stop_distance

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = high >= threshold

            # Stop definitely hit before threshold/target
            if stop_hit and not threshold_hit and not target_hit:
                return "Failure"

            # Target reached without stop also being touched
            if target_hit and not stop_hit:
                return "Success"

            # Both sides touched in same candle: order unknown
            if stop_hit and (threshold_hit or target_hit):
                return "Unknown"

            # Threshold reached: move stop to breakeven
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

    # -------------------------
    # SHORT
    # -------------------------

    elif direction == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance
        stop = opening_price + stop_distance

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Breakeven" if moved_to_breakeven else "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Breakeven"
                continue

            threshold_hit = low <= threshold

            # Stop definitely hit before threshold/target
            if stop_hit and not threshold_hit and not target_hit:
                return "Failure"

            # Target reached without stop also being touched
            if target_hit and not stop_hit:
                return "Success"

            # Both sides touched in same candle: order unknown
            if stop_hit and (threshold_hit or target_hit):
                return "Unknown"

            # Threshold reached: move stop to breakeven
            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

    else:
        return "Invalid"

    return "Neither"

In [34]:
predicted_be_results = []

for stop, target in stop_target_pairs:
    for _, prediction in resolved_predictions.iterrows():
        date = prediction["Date"]
        direction = prediction["Bias"]

        session = candles[
            candles["Date"] == date.date()
        ].sort_values("timestamp").reset_index(drop=True)

        if session.empty:
            continue

        opening_price = session.iloc[0]["open"]

        outcome = evaluate_predicted_one_move_breakeven(
            session,
            opening_price,
            direction=direction,
            target_distance=target,
            stop_distance=stop
        )

        predicted_be_results.append({
            "Date": date,
            "Day": prediction["Day"],
            "Bias": direction,
            "Correct": prediction["Correct"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

predicted_be_results = pd.DataFrame(predicted_be_results)

print("Rows:", len(predicted_be_results))
print("Unique dates:", predicted_be_results["Date"].nunique())

print()
print(
    predicted_be_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1845
Unique dates: 205

Stop  Target  Outcome  
15    25      Success      73
              Unknown      64
              Failure      61
              Breakeven     7
25    50      Failure      84
              Success      75
              Breakeven    30
              Unknown      16
35    70      Failure      92
              Success      69
              Breakeven    40
              Unknown       4
50    70      Success      92
              Failure      87
              Breakeven    24
              Unknown       2
      80      Failure      87
              Success      83
              Breakeven    32
              Unknown       3
      100     Failure      87
              Success      63
              Breakeven    52
              Unknown       3
75    100     Success      87
              Failure      86
              Breakeven    31
              Unknown       1
      150     Failure      86
              Success      59
              Breakeven    56
              Ne

In [35]:
# Actual one-time BE hit rate by weekday
# Breakeven / Unknown / Neither excluded

resolved_predicted_be = predicted_be_results[
    predicted_be_results["Outcome"].isin(["Success", "Failure"])
].copy()

be_hit_rate_table = (
    resolved_predicted_be
    .assign(Hit=resolved_predicted_be["Outcome"].eq("Success"))
    .groupby(["Stop", "Target", "Day"])["Hit"]
    .mean()
    .mul(100)
    .unstack("Day")
)

be_hit_rate_table = be_hit_rate_table[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
]

be_hit_rate_table.index = [
    f"{stop}/{target}"
    for stop, target in be_hit_rate_table.index
]

be_hit_rate_table.index.name = "Pair"

be_hit_rate_table.round(2)

Day,Monday,Tuesday,Wednesday,Thursday,Friday
Pair,,,,,
15/25,61.54,45.16,57.58,50.00,60.00
25/50,50.00,42.42,56.76,38.24,48.15
35/70,45.45,43.24,53.12,34.48,36.67
50/70,48.65,56.41,55.56,40.00,56.25
50/80,47.22,52.78,52.94,40.00,51.72
50/100,42.42,45.16,42.86,34.38,46.15
75/100,43.24,51.35,54.29,39.39,64.52
75/150,27.59,37.93,48.39,33.33,57.69
100/150,29.03,48.39,48.57,36.36,64.29
